# BRAMASTRA dual-T4 throughput pilot

Select **GPU T4 x2** and enable Internet, then run all cells. This short pilot starts one independent training worker on each GPU and tests microbatches 1, 2, 4, 8, 16, 32, and 64 concurrently. It takes about 2 minutes plus startup.

It measures model-training throughput and sampled GPU use on random byte-token batches. It does not train the frozen K8 campaign, test learning quality, or add speech transcription. The final JSON recommends a measured batch size per GPU against a 75% utilization target, or explicitly reports when the target was not reached.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

WORK = Path('/kaggle/working')
REPO = WORK / 'bramastra-throughput-source'
GIT_URL = 'https://github.com/dhurv0045com-spec/An-Ra-the-new-AGI.git'
if not REPO.exists():
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', 'BRAMASTRA', GIT_URL, str(REPO)], check=True)
else:
    subprocess.run(['git', '-C', str(REPO), 'fetch', '--depth', '1', 'origin', 'BRAMASTRA'], check=True)
    subprocess.run(['git', '-C', str(REPO), 'checkout', '-B', 'BRAMASTRA', 'origin/BRAMASTRA'], check=True)
os.chdir(REPO)
import torch
if not torch.cuda.is_available() or torch.cuda.device_count() < 2:
    raise RuntimeError(f'Requires Kaggle GPU T4 x2; detected {torch.cuda.device_count()} CUDA GPU(s).')
print('Source revision:', subprocess.check_output(['git', 'rev-parse', '--short', 'HEAD'], text=True).strip())
print('GPUs:', [torch.cuda.get_device_name(i) for i in range(2)])

In [ ]:
REPORT = WORK / 'dual_gpu_throughput.json'
if REPORT.exists():
    REPORT.unlink()
command = [
    sys.executable, '-m', 'bramastra_lab.research.campaigns.gpu_throughput',
    '--devices', '0,1', '--batch-sizes', '1,2,4,8,16,32,64',
    '--seconds-per-case', '20', '--sequence-length', '512',
    '--target-utilization', '75', '--out', str(REPORT),
]
completed = subprocess.run(command, cwd=REPO, text=True, check=False)
if not REPORT.exists():
    raise RuntimeError(f'Throughput pilot failed before writing its report (exit {completed.returncode}).')
print(REPORT.read_text(encoding='utf-8'))
print('Download this report from Kaggle Output:', REPORT)
if completed.returncode:
    raise RuntimeError(f'Throughput pilot completed with errors (exit {completed.returncode}); inspect the report above.')